In [1]:
!pip install vaderSentiment -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.4 MB/s eta 0:00:00


In [2]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

print(analyzer.polarity_scores("The app is good."))

{'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.4404}


In [3]:
for text in ["The app is good.",
             "The app is GOOD.",
             "The app is good!!!",
             "The app is very good.",
             "The app is not good."]:
    score = analyzer.polarity_scores(text)["compound"]
    print(f"{score:+.4f} {text}")

+0.4404 The app is good.
+0.5622 The app is GOOD.
+0.5826 The app is good!!!
+0.4927 The app is very good.
-0.3412 The app is not good.


In [4]:
for text in ["The app is not good.",
             "The app is not good at all.",
             "Wonderful. Third outage this week.",
             "App dey crash anytime I wan transfer.",
             "Abeg make una fix this app."]:
    score = analyzer.polarity_scores(text)["compound"]
    print(f"{score:+.4f} {text}")

-0.3412 The app is not good.
-0.3412 The app is not good at all.
+0.5719 Wonderful. Third outage this week.
-0.4019 App dey crash anytime I wan transfer.
+0.0000 Abeg make una fix this app.


In [5]:
def vader_label(text, pos_cut=0.05, neg_cut=-0.05):
    """Turn VADER's compound score into one of our three labels."""
    score = analyzer.polarity_scores(text)["compound"]
    if score >= pos_cut:
        return "positive"
    if score <= neg_cut:
        return "negative"
    return "neutral"

print(vader_label("The app is not good at all."))
print(vader_label("Transfer was fast, I like am."))
print(vader_label("It works."))

negative
positive
neutral


In [6]:
  TEST_SET = [
    ("The app is fast and the transfer went through immediately.", "positive"),
    ("Transfer was fast, I like am.", "positive"),
    ("Best banking app I have used, no stress at all.", "positive"),
    ("Customer care called me back within 5 minutes. Impressed.", "positive"),
    ("Finally they fixed the login issue. Working well now.", "positive"),
    ("The transfer failed but support sorted it quickly.", "positive"),
    ("App dey crash anytime I wan transfer.", "negative"),
    ("They charged me twice and nobody is responding.", "negative"),
    ("The app is not good at all.", "negative"),
    ("Abeg make una fix this app.", "negative"),
    ("Wonderful. Third outage this week.", "negative"),
    ("I have been on hold for 40 minutes. Terrible service.", "negative"),
    ("Why is the OTP not coming? This is frustrating.", "negative"),
    ("Good app, but the fees are too high.", "negative"),
    ("I downloaded the app yesterday.", "neutral"),
    ("Does this app work on Android 14?", "neutral"),
    ("It works.", "neutral"),
    ("Please add dark mode in the next update.", "neutral"),
    ("The app size is 85MB.", "neutral"),
    ("I use this app for my business account.", "neutral"),
]

LABELS = ["positive", "neutral", "negative"]

print(f"{len(TEST_SET)} items")

20 items


In [7]:
from collections import Counter

print(Counter(label for _, label in TEST_SET))

Counter({'negative': 8, 'positive': 6, 'neutral': 6})


In [8]:
def always_positive(text):
    return "positive"

gold = [label for _, label in TEST_SET]
pred = [always_positive(text) for text, _ in TEST_SET]

correct = sum(g == p for g, p in zip(gold, pred))

print(f"Baseline accuracy: {correct}/{len(gold)} = {correct/len(gold):.0%}")

Baseline accuracy: 6/20 = 30%


In [9]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(predict_fn, name):
    """Score any labelling function against the fixed test set."""
    gold = [label for _, label in TEST_SET]
    pred = [predict_fn(text) for text, _ in TEST_SET]

    correct = sum(g == p for g, p in zip(gold, pred))

    print(f"\n===== {name} =====")
    print(f"Accuracy: {correct}/{len(gold)} = {correct/len(gold):.0%}\n")

    cm = confusion_matrix(gold, pred, labels=LABELS)

    print(f"{'true \\ said':14}" + "".join(f"{l:>10}" for l in LABELS))

    for lab, row in zip(LABELS, cm):
        print(f"{lab:14}" + "".join(f"{v:>10}" for v in row))

    print()

    print(classification_report(
        gold,
        pred,
        labels=LABELS,
        digits=2,
        zero_division=0
    ))

    return pred

evaluate(always_positive, "BASELINE - always positive")


===== BASELINE - always positive =====
Accuracy: 6/20 = 30%

true \ said     positive   neutral  negative
positive               6         0         0
neutral                6         0         0
negative               8         0         0

              precision    recall  f1-score   support

    positive       0.30      1.00      0.46         6
     neutral       0.00      0.00      0.00         6
    negative       0.00      0.00      0.00         8

    accuracy                           0.30        20
   macro avg       0.10      0.33      0.15        20
weighted avg       0.09      0.30      0.14        20



['positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive']

In [10]:
evaluate(vader_label, "VADER - default thresholds")


===== VADER - default thresholds =====
Accuracy: 15/20 = 75%

true \ said     positive   neutral  negative
positive               5         1         0
neutral                1         5         0
negative               2         1         5

              precision    recall  f1-score   support

    positive       0.62      0.83      0.71         6
     neutral       0.71      0.83      0.77         6
    negative       1.00      0.62      0.77         8

    accuracy                           0.75        20
   macro avg       0.78      0.76      0.75        20
weighted avg       0.80      0.75      0.75        20



['neutral',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'negative',
 'negative',
 'negative',
 'neutral',
 'positive',
 'negative',
 'negative',
 'positive',
 'neutral',
 'neutral',
 'neutral',
 'positive',
 'neutral',
 'neutral']

In [11]:
def show_mistakes(predict_fn, name):
    print(f"--- {name}: what it got wrong ---")

    for text, gold_label in TEST_SET:
        guess = predict_fn(text)

        if guess != gold_label:
            comp = analyzer.polarity_scores(text)["compound"]
            print(
                f"said {guess:9} want {gold_label:9} "
                f"({comp:+.3f}) {text[:50]}"
            )

show_mistakes(vader_label, "VADER")

--- VADER: what it got wrong ---
said neutral   want positive  (+0.000) The app is fast and the transfer went through imme
said neutral   want negative  (+0.000) Abeg make una fix this app.
said positive  want negative  (+0.572) Wonderful. Third outage this week.
said positive  want negative  (+0.238) Good app, but the fees are too high.
said positive  want neutral   (+0.318) Please add dark mode in the next update.


In [12]:
def wide_vader(text):
    return vader_label(text, pos_cut=0.25, neg_cut=-0.25)

evaluate(wide_vader, "VADER - wider neutral band")


===== VADER - wider neutral band =====
Accuracy: 14/20 = 70%

true \ said     positive   neutral  negative
positive               5         1         0
neutral                1         5         0
negative               1         3         4

              precision    recall  f1-score   support

    positive       0.71      0.83      0.77         6
     neutral       0.56      0.83      0.67         6
    negative       1.00      0.50      0.67         8

    accuracy                           0.70        20
   macro avg       0.76      0.72      0.70        20
weighted avg       0.78      0.70      0.70        20



['neutral',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'negative',
 'neutral',
 'negative',
 'neutral',
 'positive',
 'negative',
 'negative',
 'neutral',
 'neutral',
 'neutral',
 'neutral',
 'positive',
 'neutral',
 'neutral']

In [13]:
!pip install transformers torch -q

In [14]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
)

print(classifier("The app is not good at all."))
print(classifier("Abeg make una fix this app."))

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[{'label': 'negative', 'score': 0.9382528066635132}]
[{'label': 'negative', 'score': 0.7564468383789062}]


In [15]:
def model_label(text):
    """Map this model's output onto our three labels."""
    raw = classifier(text)[0]["label"].lower()

    mapping = {
        "label_0": "negative",
        "label_1": "neutral",
        "label_2": "positive",
        "negative": "negative",
        "neutral": "neutral",
        "positive": "positive",
        "neg": "negative",
        "neu": "neutral",
        "pos": "positive",
    }

    return mapping.get(raw, "neutral")

print(model_label("Abeg make una fix this app."))

negative


In [16]:
evaluate(model_label, "TRANSFORMER - twitter-roberta")

show_mistakes(model_label, "TRANSFORMER")


===== TRANSFORMER - twitter-roberta =====
Accuracy: 17/20 = 85%

true \ said     positive   neutral  negative
positive               5         1         0
neutral                1         5         0
negative               1         0         7

              precision    recall  f1-score   support

    positive       0.71      0.83      0.77         6
     neutral       0.83      0.83      0.83         6
    negative       1.00      0.88      0.93         8

    accuracy                           0.85        20
   macro avg       0.85      0.85      0.85        20
weighted avg       0.86      0.85      0.85        20

--- TRANSFORMER: what it got wrong ---
said neutral   want positive  (+0.340) The transfer failed but support sorted it quickly.
said positive  want negative  (+0.572) Wonderful. Third outage this week.
said positive  want neutral   (+0.000) It works.


In [17]:
from sklearn.metrics import f1_score, recall_score

approaches = [
    ("Baseline", always_positive),
    ("VADER", vader_label),
    ("VADER wide", wide_vader),
    ("Transformer", model_label),
]

gold = [label for _, label in TEST_SET]

print(f"{'Approach':14}{'Accuracy':>10}{'Macro F1':>10}{'Neg recall':>12}")

for name, fn in approaches:
    pred = [fn(text) for text, _ in TEST_SET]

    acc = sum(g == p for g, p in zip(gold, pred)) / len(gold)

    mf1 = f1_score(
        gold,
        pred,
        average="macro",
        zero_division=0
    )

    rec = recall_score(
        gold,
        pred,
        labels=["negative"],
        average="macro",
        zero_division=0
    )

    print(f"{name:14}{acc:>10.0%}{mf1:>10.2f}{rec:>12.2f}")

Approach        Accuracy  Macro F1  Neg recall
Baseline             30%      0.15        0.00
VADER                75%      0.75        0.62
VADER wide           70%      0.70        0.50
Transformer          85%      0.85        0.88


In [18]:
my_reviews = [
    "The app is very fast and easy to use.",
    "This app dey stress me every time I wan transfer.",
    "Wonderful, another outage. Just what I needed today.",
    "I use the app to check my account balance.",
    "The app is useful, but the transfer problems are frustrating.",
]

for text in my_reviews:
    v = vader_label(text)
    score = analyzer.polarity_scores(text)["compound"]
    print(f"VADER: {v:9} ({score:+.3f}) {text}")

VADER: positive  (+0.488) The app is very fast and easy to use.
VADER: negative  (-0.421) This app dey stress me every time I wan transfer.
VADER: positive  (+0.572) Wonderful, another outage. Just what I needed today.
VADER: neutral   (+0.000) I use the app to check my account balance.
VADER: negative  (-0.754) The app is useful, but the transfer problems are frustrating.


# **`VADER Failure: Sarcasm`**

VADER classified “Wonderful, another outage. Just what I needed today.” as positive because it focused on positive words like “Wonderful.” However, the sentence is sarcastic and expresses frustration about another outage. This shows that VADER struggles to understand sarcasm and the context of a sentence.

# **THE APPROACH I WOULD CHOOSE**

I would choose the transformer model because it achieved 85% accuracy, a macro F1 score of 0.85, and 0.88 recall for negative reviews, outperforming the other approaches on this test set. However, I would test it on more customer reviews, including Pidgin, sarcasm, and mixed sentiments, before using it in a real banking application.